In [ ]:
import re
import os
import unicodedata
import urllib3
import zipfile
import shutil
import numpy as np
import pandas as pd
import torch
from collections import Counter
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
num_samples = 33000

In [ ]:
!wget -c http://www.manythings.org/anki/fra-eng.zip && unzip -o fra-eng.zip

--2024-09-05 14:55:29--  http://www.manythings.org/anki/fra-eng.zip
Resolving www.manythings.org (www.manythings.org)... 173.254.30.110
Connecting to www.manythings.org (www.manythings.org)|173.254.30.110|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7943074 (7.6M) [application/zip]
Saving to: ‘fra-eng.zip’

fra-eng.zip         100%[===================>]   7.57M  14.5MB/s    in 0.5s    

2024-09-05 14:55:30 (14.5 MB/s) - ‘fra-eng.zip’ saved [7943074/7943074]

Archive:  fra-eng.zip
  inflating: _about.txt              
  inflating: fra.txt                 


In [ ]:
def unicode_to_ascii(s):
  # Remove French accents
  # Example: 'déjà diné' -> deja dine
  return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

In [ ]:
def preprocess_sentence(sent):
  # Call the accent removal function
  sent = unicode_to_ascii(sent.lower())

  # Add spaces between words and punctuation marks.
  # Ex) "he is a boy." => "he is a boy ."
  sent = re.sub(r"([?.!,¿])", r" \1", sent)

  # Convert all characters except (a-z, A-Z, ".", "?", "!", ",") to spaces.
  sent = re.sub(r"[^a-zA-Z!.?]+", r" ", sent)

  # Replace multiple spaces with a single space
  sent = re.sub(r"\s+", " ", sent)
  return sent

In [ ]:
def load_preprocessed_data():
  encoder_input, decoder_input, decoder_target = [], [], []

  with open("fra.txt", "r") as lines:
    for i, line in enumerate(lines):
      # Separate the source data and target data
      src_line, tar_line, _ = line.strip().split('\t')

      # Preprocess the source data
      src_line = [w for w in preprocess_sentence(src_line).split()]

      # Preprocess the target data
      tar_line = preprocess_sentence(tar_line)
      tar_line_in = [w for w in ("<sos> " + tar_line).split()]
      tar_line_out = [w for w in (tar_line + " <eos>").split()]

      encoder_input.append(src_line)
      decoder_input.append(tar_line_in)
      decoder_target.append(tar_line_out)

      if i == num_samples - 1:
        break

  return encoder_input, decoder_input, decoder_target

In [ ]:
# Preprocessing test
en_sent = u"Have you had dinner?"
fr_sent = u"Avez-vous déjà diné?"

print('English sentence before preprocessing :', en_sent)
print('English sentence after preprocessing :',preprocess_sentence(en_sent))
print('French sentence before preprocessing :', fr_sent)
print('French sentence after preprocessing :', preprocess_sentence(fr_sent))

English sentence before preprocessing : Have you had dinner?
English sentence after preprocessing : have you had dinner ?
French sentence before preprocessing : Avez-vous déjà diné?
French sentence after preprocessing : avez vous deja dine ?


In [ ]:
sents_en_in, sents_fra_in, sents_fra_out = load_preprocessed_data()

In [ ]:
sents_en_in, sents_fra_in, sents_fra_out = load_preprocessed_data()
print('Encoder input :',sents_en_in[:5])
print('Decoder input :',sents_fra_in[:5])
print('Decoder label :',sents_fra_out[:5])

Encoder input : [['go', '.'], ['go', '.'], ['go', '.'], ['go', '.'], ['hi', '.']]
Decoder input : [['<sos>', 'va', '!'], ['<sos>', 'marche', '.'], ['<sos>', 'en', 'route', '!'], ['<sos>', 'bouge', '!'], ['<sos>', 'salut', '!']]
Decoder label : [['va', '!', '<eos>'], ['marche', '.', '<eos>'], ['en', 'route', '!', '<eos>'], ['bouge', '!', '<eos>'], ['salut', '!', '<eos>']]


In [ ]:
def build_vocab(sents):
  word_list = []

  for sent in sents:
      for word in sent:
        word_list.append(word)

  # Compute the frequency of each word and sort by descending frequency
  word_counts = Counter(word_list)
  vocab = sorted(word_counts, key=word_counts.get, reverse=True)

  word_to_index = {}
  word_to_index['<PAD>'] = 0
  word_to_index['<UNK>'] = 1

  # Assign lower integer IDs to more frequent words
  for index, word in enumerate(vocab) :
    word_to_index[word] = index + 2

  return word_to_index

In [ ]:
src_vocab = build_vocab(sents_en_in)
tar_vocab = build_vocab(sents_fra_in + sents_fra_out)

src_vocab_size = len(src_vocab)
tar_vocab_size = len(tar_vocab)
print("English vocabulary size : {:d}, French vocabulary size : {:d}".format(src_vocab_size, tar_vocab_size))

English vocabulary size : 4486, French vocabulary size : 7879


In [ ]:
index_to_src = {v: k for k, v in src_vocab.items()}
index_to_tar = {v: k for k, v in tar_vocab.items()}

def texts_to_sequences(sents, word_to_index):
  encoded_X_data = []
  for sent in tqdm(sents):
    index_sequences = []
    for word in sent:
      try:
          index_sequences.append(word_to_index[word])
      except KeyError:
          index_sequences.append(word_to_index['<UNK>'])
    encoded_X_data.append(index_sequences)
  return encoded_X_data

In [ ]:
encoder_input = texts_to_sequences(sents_en_in, src_vocab)
decoder_input = texts_to_sequences(sents_fra_in, tar_vocab)
decoder_target = texts_to_sequences(sents_fra_out, tar_vocab)

100%|██████████| 33000/33000 [00:00<00:00, 115858.16it/s]


In [ ]:
# Print sentences before and after integer encoding for the top 5 samples
# Since this is encoder input, it does not include <sos> or <eos>
for i, (item1, item2) in zip(range(5), zip(sents_en_in, encoder_input)):
    print(f"Index: {i}, Before integer encoding: {item1}, After integer encoding: {item2}")

Index: 0, Before integer encoding: ['go', '.'], After integer encoding: [27, 2]
Index: 1, Before integer encoding: ['go', '.'], After integer encoding: [27, 2]
Index: 2, Before integer encoding: ['go', '.'], After integer encoding: [27, 2]
Index: 3, Before integer encoding: ['go', '.'], After integer encoding: [27, 2]
Index: 4, Before integer encoding: ['hi', '.'], After integer encoding: [736, 2]


In [ ]:
def pad_sequences(sentences, max_len=None):
    # If max_len is not given, pad to the maximum length in the data
    if max_len is None:
        max_len = max([len(sentence) for sentence in sentences])

    features = np.zeros((len(sentences), max_len), dtype=int)
    for index, sentence in enumerate(sentences):
        if len(sentence) != 0:
            features[index, :len(sentence)] = np.array(sentence)[:max_len]
    return features

In [ ]:
encoder_input = pad_sequences(encoder_input)
decoder_input = pad_sequences(decoder_input)
decoder_target = pad_sequences(decoder_target)

In [ ]:
print('Shape of encoder input :',encoder_input.shape)
print('Shape of decoder input :',decoder_input.shape)
print('Shape of decoder labels :',decoder_target.shape)

Shape of encoder input : (33000, 7)
Shape of decoder input : (33000, 16)
Shape of decoder labels : (33000, 16)


In [ ]:
indices = np.arange(encoder_input.shape[0])
np.random.shuffle(indices)
print('Random sequence :',indices)

Random sequence : [12479  9365 20901 ... 11254 18145 23491]


In [ ]:
encoder_input = encoder_input[indices]
decoder_input = decoder_input[indices]
decoder_target = decoder_target[indices]

In [ ]:
print([index_to_src[word] for word in encoder_input[30997]])
print([index_to_tar[word] for word in decoder_input[30997]])
print([index_to_tar[word] for word in decoder_target[30997]])

['how', 's', 'your', 'sister', '?', '<PAD>', '<PAD>']
['<sos>', 'comment', 'va', 'ta', 's', 'ur', '?', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>']
['comment', 'va', 'ta', 's', 'ur', '?', '<eos>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>', '<PAD>']


In [ ]:
n_of_val = int(33000*0.1)
print('Number of validation samples :',n_of_val)

Number of validation samples : 3300


In [ ]:
encoder_input_train = encoder_input[:-n_of_val]
decoder_input_train = decoder_input[:-n_of_val]
decoder_target_train = decoder_target[:-n_of_val]

encoder_input_test = encoder_input[-n_of_val:]
decoder_input_test = decoder_input[-n_of_val:]
decoder_target_test = decoder_target[-n_of_val:]

In [ ]:
print('Shape of training source data :',encoder_input_train.shape)
print('Shape of training target data :',decoder_input_train.shape)
print('Shape of training target labels :',decoder_target_train.shape)
print('Shape of test source data :',encoder_input_test.shape)
print('Shape of test target data :',decoder_input_test.shape)
print('Shape of test target labels :',decoder_target_test.shape)

Shape of training source data : (29700, 7)
Shape of training target data : (29700, 16)
Shape of training target labels : (29700, 16)
Shape of test source data : (3300, 7)
Shape of test target data : (3300, 16)
Shape of test target labels : (3300, 16)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

embedding_dim = 256
hidden_units = 256

In [ ]:
class Encoder(nn.Module):
    def __init__(self, src_vocab_size, embedding_dim, hidden_units):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(src_vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_units, batch_first=True)

    def forward(self, x):
        # x.shape == (batch_size, seq_len, embedding_dim)
        x = self.embedding(x)
        # hidden.shape == (1, batch_size, hidden_units), cell.shape == (1, batch_size, hidden_units)
        outputs, (hidden, cell) = self.lstm(x)
        return outputs, hidden, cell

In [ ]:
class Decoder(nn.Module):
    def __init__(self, tar_vocab_size, embedding_dim, hidden_units):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(tar_vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim + hidden_units, hidden_units, batch_first=True)
        self.fc = nn.Linear(hidden_units, tar_vocab_size)

    def forward(self, x, encoder_outputs, hidden, cell):
        """
        x: (batch_size, target_seq_len)
        encoder_outputs: (batch_size, source_seq_len, hidden_units)
        hidden: (1, batch_size, hidden_units)
        cell: (1, batch_size, hidden_units)

        Dot-product attention is computed at every decoder time step.
        The query is the current decoder hidden state, initialized from the
        encoder final hidden state and then updated after each decoder step.
        """
        outputs = []
        target_seq_len = x.size(1)

        for t in range(target_seq_len):
            # Current decoder input token: (batch_size, 1)
            input_t = x[:, t].unsqueeze(1)

            # Embedded decoder input: (batch_size, 1, embedding_dim)
            embedded = self.embedding(input_t)

            # Dot-product attention.
            # query.shape: (batch_size, hidden_units, 1)
            query = hidden[-1].unsqueeze(2)

            # attention_scores.shape: (batch_size, source_seq_len, 1)
            attention_scores = torch.bmm(encoder_outputs, query)

            # Normalize over the source sequence dimension.
            # attention_weights.shape: (batch_size, source_seq_len, 1)
            attention_weights = torch.softmax(attention_scores, dim=1)

            # context_vector.shape: (batch_size, 1, hidden_units)
            context_vector = torch.bmm(attention_weights.transpose(1, 2), encoder_outputs)

            # Decoder input for this step: (batch_size, 1, embedding_dim + hidden_units)
            decoder_input = torch.cat((embedded, context_vector), dim=2)

            # Run one decoder step.
            # output.shape: (batch_size, 1, hidden_units)
            output, (hidden, cell) = self.lstm(decoder_input, (hidden, cell))

            # output.shape: (batch_size, 1, tar_vocab_size)
            output = self.fc(output)
            outputs.append(output)

        # outputs.shape: (batch_size, target_seq_len, tar_vocab_size)
        outputs = torch.cat(outputs, dim=1)
        return outputs, hidden, cell

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        encoder_outputs, hidden, cell = self.encoder(src)
        output, _, _ = self.decoder(trg, encoder_outputs, hidden, cell)
        return output

encoder = Encoder(src_vocab_size, embedding_dim, hidden_units)
decoder = Decoder(tar_vocab_size, embedding_dim, hidden_units)
model = Seq2Seq(encoder, decoder)

loss_function = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters())

In [ ]:
def evaluation(model, dataloader, loss_function, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for encoder_inputs, decoder_inputs, decoder_targets in dataloader:
            encoder_inputs = encoder_inputs.to(device)
            decoder_inputs = decoder_inputs.to(device)
            decoder_targets = decoder_targets.to(device)

            # Forward pass
            # outputs.shape == (batch_size, seq_len, tar_vocab_size)
            outputs = model(encoder_inputs, decoder_inputs)

            # Compute the loss
            # The shape of outputs.view(-1, outputs.size(-1)) is (batch_size * seq_len, tar_vocab_size)
            # The shape of decoder_targets.view(-1) is (batch_size * seq_len)
            loss = loss_function(outputs.view(-1, outputs.size(-1)), decoder_targets.view(-1))
            total_loss += loss.item()

            # Compute accuracy (excluding padding tokens)
            mask = decoder_targets != 0
            total_correct += ((outputs.argmax(dim=-1) == decoder_targets) * mask).sum().item()
            total_count += mask.sum().item()

    return total_loss / len(dataloader), total_correct / total_count

In [ ]:
encoder_input_train_tensor = torch.tensor(encoder_input_train, dtype=torch.long)
decoder_input_train_tensor = torch.tensor(decoder_input_train, dtype=torch.long)
decoder_target_train_tensor = torch.tensor(decoder_target_train, dtype=torch.long)

encoder_input_test_tensor = torch.tensor(encoder_input_test, dtype=torch.long)
decoder_input_test_tensor = torch.tensor(decoder_input_test, dtype=torch.long)
decoder_target_test_tensor = torch.tensor(decoder_target_test, dtype=torch.long)

# Create the dataset and dataloader
batch_size = 128

train_dataset = TensorDataset(encoder_input_train_tensor, decoder_input_train_tensor, decoder_target_train_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

valid_dataset = TensorDataset(encoder_input_test_tensor, decoder_input_test_tensor, decoder_target_test_tensor)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

# Training setup
num_epochs = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(4486, 256, padding_idx=0)
    (lstm): LSTM(256, 256, batch_first=True)
  )
  (decoder): Decoder(
    (embedding): Embedding(7879, 256, padding_idx=0)
    (lstm): LSTM(512, 256, batch_first=True)
    (fc): Linear(in_features=256, out_features=7879, bias=True)
    (softmax): Softmax(dim=2)
  )
)

In [ ]:
# Training loop
best_val_loss = float('inf')

for epoch in range(num_epochs):
    # Training mode
    model.train()

    for encoder_inputs, decoder_inputs, decoder_targets in train_dataloader:
        encoder_inputs = encoder_inputs.to(device)
        decoder_inputs = decoder_inputs.to(device)
        decoder_targets = decoder_targets.to(device)

        # Initialize gradients
        optimizer.zero_grad()

        # Forward pass
        # outputs.shape == (batch_size, seq_len, tar_vocab_size)
        outputs = model(encoder_inputs, decoder_inputs)

        # Compute the loss and perform backpropagation
        # The shape of outputs.view(-1, outputs.size(-1)) is (batch_size * seq_len, tar_vocab_size)
        # The shape of decoder_targets.view(-1) is (batch_size * seq_len)
        loss = loss_function(outputs.view(-1, outputs.size(-1)), decoder_targets.view(-1))
        loss.backward()

        # Update weights
        optimizer.step()

    train_loss, train_acc = evaluation(model, train_dataloader, loss_function, device)
    valid_loss, valid_acc = evaluation(model, valid_dataloader, loss_function, device)

    print(f'Epoch: {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Valid Loss: {valid_loss:.4f} | Valid Acc: {valid_acc:.4f}')

    # Save a checkpoint when the validation loss reaches a new minimum
    if valid_loss < best_val_loss:
        print(f'Validation loss improved from {best_val_loss:.4f} to {valid_loss:.4f}. Saving checkpoint.')
        best_val_loss = valid_loss
        torch.save(model.state_dict(), 'best_model_checkpoint.pth')

Epoch: 1/30 | Train Loss: 2.2343 | Train Acc: 0.6188 | Valid Loss: 2.4480 | Valid Acc: 0.6076
Validation loss improved from inf to 2.4480. Saving checkpoint.
Epoch: 2/30 | Train Loss: 1.7689 | Train Acc: 0.6671 | Valid Loss: 2.1103 | Valid Acc: 0.6400
Validation loss improved from 2.4480 to 2.1103. Saving checkpoint.
Epoch: 3/30 | Train Loss: 1.4138 | Train Acc: 0.7148 | Valid Loss: 1.8879 | Valid Acc: 0.6668
Validation loss improved from 2.1103 to 1.8879. Saving checkpoint.
Epoch: 4/30 | Train Loss: 1.1270 | Train Acc: 0.7568 | Valid Loss: 1.7234 | Valid Acc: 0.6875
Validation loss improved from 1.8879 to 1.7234. Saving checkpoint.
Epoch: 5/30 | Train Loss: 0.9020 | Train Acc: 0.7987 | Valid Loss: 1.6124 | Valid Acc: 0.7034
Validation loss improved from 1.7234 to 1.6124. Saving checkpoint.
Epoch: 6/30 | Train Loss: 0.7208 | Train Acc: 0.8343 | Valid Loss: 1.5204 | Valid Acc: 0.7130
Validation loss improved from 1.6124 to 1.5204. Saving checkpoint.
Epoch: 7/30 | Train Loss: 0.5799 | Tr

In [ ]:
# Load the model
model.load_state_dict(torch.load('best_model_checkpoint.pth'))

# Move the model to the device.
model.to(device)

# Compute accuracy and loss on the validation data
val_loss, val_accuracy = evaluation(model, valid_dataloader, loss_function, device)

print(f'Best model validation loss: {val_loss:.4f}')
print(f'Best model validation accuracy: {val_accuracy:.4f}')

Best model validation loss: 1.4097
Best model validation accuracy: 0.7371


<ipython-input-32-ccccf4da9b07>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model_checkpoint.pth'))


In [ ]:
print(tar_vocab['<sos>'])
print(tar_vocab['<eos>'])

3
4


In [ ]:
index_to_src = {v: k for k, v in src_vocab.items()}
index_to_tar = {v: k for k, v in tar_vocab.items()}

# Convert the source integer sequence to a text sequence
def seq_to_src(input_seq):
  sentence = ''
  for encoded_word in input_seq:
    if(encoded_word != 0):
      sentence = sentence + index_to_src[encoded_word] + ' '
  return sentence

# Convert the target integer sequence to a text sequence
def seq_to_tar(input_seq):
  sentence = ''
  for encoded_word in input_seq:
    if(encoded_word != 0 and encoded_word != tar_vocab['<sos>'] and encoded_word != tar_vocab['<eos>']):
      sentence = sentence + index_to_tar[encoded_word] + ' '
  return sentence

In [ ]:
print(encoder_input_test[25])
print(decoder_input_test[25])
print(decoder_target_test[25])

[ 25 563   6   2   0   0   0]
[   3   46   79 1791    8    2    0    0    0    0    0    0    0    0
    0    0]
[  46   79 1791    8    2    4    0    0    0    0    0    0    0    0
    0    0]


In [ ]:
def decode_sequence(input_seq, model, src_vocab_size, tar_vocab_size, max_output_len, int_to_src_token, int_to_tar_token):
    encoder_inputs = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)

    # Set the initial encoder state
    encoder_outputs, hidden, cell = model.encoder(encoder_inputs)

    # Set the start token <sos> as the first decoder input
    # unsqueeze(0) adds the batch dimension.
    decoder_input = torch.tensor([3], dtype=torch.long).unsqueeze(0).to(device)

    decoded_tokens = []

    # Each loop iteration corresponds to one decoder time step
    for _ in range(max_output_len):
        output, hidden, cell = model.decoder(decoder_input, encoder_outputs, hidden, cell)

        # Get the predicted token index
        output_token = output.argmax(dim=-1).item()

        # End token <eos>
        if output_token == 4:
            break

        # Accumulate the token IDs at each time step in decoded_tokens and return them as the final translation sequence.
        decoded_tokens.append(output_token)

        # Use the current prediction as the input for the next time step.
        decoder_input = torch.tensor([output_token], dtype=torch.long).unsqueeze(0).to(device)

    return ' '.join(int_to_tar_token[token] for token in decoded_tokens)

In [ ]:
for seq_index in [3, 50, 100, 300, 1001]:
  input_seq = encoder_input_train[seq_index]
  translated_text = decode_sequence(input_seq, model, src_vocab_size, tar_vocab_size, 20, index_to_src, index_to_tar)

  print("Input sentence :",seq_to_src(encoder_input_train[seq_index]))
  print("Reference sentence :",seq_to_tar(decoder_input_train[seq_index]))
  print("Translated sentence :",translated_text)
  print("-"*50)

Input sentence : read the meter . 
Reference sentence : lis le compteur . 
Translated sentence : lis le compteur .
--------------------------------------------------
Input sentence : we ll walk . 
Reference sentence : nous marcherons . 
Translated sentence : nous marcherons .
--------------------------------------------------
Input sentence : remain seated . 
Reference sentence : reste assise . 
Translated sentence : restez assise .
--------------------------------------------------
Input sentence : tom hates opera . 
Reference sentence : tom deteste l opera . 
Translated sentence : tom deteste l opera .
--------------------------------------------------
Input sentence : i never win . 
Reference sentence : jamais je ne l emporte . 
Translated sentence : je ne l emporte jamais .
--------------------------------------------------


In [ ]:
for seq_index in [3, 50, 100, 300, 1001]:
  input_seq = encoder_input_test[seq_index]
  translated_text = decode_sequence(input_seq, model, src_vocab_size, tar_vocab_size, 20, index_to_src, index_to_tar)

  print("Input sentence :",seq_to_src(encoder_input_test[seq_index]))
  print("Reference sentence :",seq_to_tar(decoder_input_test[seq_index]))
  print("Translated sentence :",translated_text)
  print("-"*50)

Input sentence : these are genuine . 
Reference sentence : ceux ci sont veritables . 
Translated sentence : les flics sont parfaits .
--------------------------------------------------
Input sentence : don t give me that ! 
Reference sentence : ne me donne pas ca ! 
Translated sentence : ne m mets pas ca !
--------------------------------------------------
Input sentence : don t do that . 
Reference sentence : ne fais pas ca . 
Translated sentence : ne fais pas ca .
--------------------------------------------------
Input sentence : can i make copies ? 
Reference sentence : puis je faire des copies ? 
Translated sentence : puis je effectuer des copies ?
--------------------------------------------------
Input sentence : what have you got ? 
Reference sentence : qu avez vous ? 
Translated sentence : qu avez vous ?
--------------------------------------------------
